# 전체 Attention 헤드의 기울기 탐색과 독립 검증

**질문:** 기존 실험이 레이어 6개만 골라 헤드를 평가하면서 한국어에 민감한 헤드를 놓쳤는가?

같은 `Qwen/Qwen2.5-1.5B-Instruct` FP16 모델과 저장된 `pilot_samples.json`을 사용합니다. 각 Query 헤드의 출력에 값이 1인 임시 게이트를 곱하고, 한국어·영어 손실을 게이트로 미분합니다. `-∂평균 NLL/∂게이트`가 클수록 해당 헤드를 조금 줄일 때 손실이 늘어날 것으로 예상합니다. 이는 **국소 근사치**이므로 최종 판단에는 게이트를 0으로 만들어 PPL을 다시 측정합니다.

- **탐색:** 평가 묶음 1의 언어별 2048토큰으로 28개 레이어의 모든 Query 헤드를 한 번에 기울기로 평가합니다. 한국어 기울기 중요도가 양수인 것 중 `한국어 중요도 - 영어 중요도` 상위 4개와, 이전에 개별 평가하지 않은 레이어의 상위 4개를 고릅니다. 겹치면 한 번만 검증합니다.
- **검증:** 선택된 헤드만 평가 묶음 2·3의 각 8192토큰에서 하나씩 0으로 만듭니다. 각 묶음에서 한국어 PPL 증가율과 `S = 한국어 증가율 - 영어 증가율`이 모두 양수인지 확인합니다. 후보 수와 평가 토큰 수는 결과를 보기 전에 고정합니다.
- **한계:** 두 언어의 평가 말뭉치가 다릅니다. 기울기 순위는 마스킹 효과 또는 양자화 민감도를 보장하지 않습니다. 이 실험은 FP16 모델의 헤드 민감도 분석이며 8bit 보존의 효과를 측정하지 않습니다.

이 파일은 로컬 `llm-quantization-lab`에 저장된 Colab용 노트북입니다. 새 GPU 런타임에서 순서대로 실행하고 업로드 창에서 `pilot_results/pilot_samples.json` 하나를 선택하세요. 설치 셀이나 Google Drive 연결은 없습니다. 결과 CSV는 브라우저로 다운로드하며, **노트북에는 실행 결과가 포함되지 않습니다.**


In [ ]:
import csv
import hashlib
import json
import math
from datetime import datetime, timezone

import torch
import transformers
from google.colab import files
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU 런타임을 사용하세요.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEQ_LEN = 512
SCREEN_TOKENS = 4 * SEQ_LEN
VALIDATION_TOKENS = 16 * SEQ_LEN
TOP_GLOBAL = 4
TOP_NEW_LAYERS = 4
# 이전 head_screen.csv에서 개별 헤드를 평가한 레이어입니다.
PREVIOUSLY_SCREENED_LAYERS = {0, 1, 2, 3, 5, 18}

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("pilot_results/pilot_samples.json 하나만 업로드하세요.")
sample_bytes = next(iter(uploaded.values()))
if hashlib.sha256(sample_bytes).hexdigest() != "762e9a33457824598b73e49812666c3fb042f25df24a09db094ffc2a04590ed6":
    raise ValueError("저장된 예비 실험의 pilot_samples.json과 해시가 다릅니다.")
samples = json.loads(sample_bytes.decode("utf-8"))
if set(samples.get("eval_groups", {})) != {"ko", "en"} or any(
    len(samples["eval_groups"][lang]) != 3 for lang in ("ko", "en")
):
    raise ValueError("한국어·영어 평가 묶음이 각각 3개여야 합니다.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0").eval()
model.requires_grad_(False)
model.config.use_cache = False
N_LAYERS = len(model.model.layers)
N_HEADS = model.config.num_attention_heads
HEAD_DIM = getattr(model.config, "head_dim", None) or model.config.hidden_size // N_HEADS
assert max(PREVIOUSLY_SCREENED_LAYERS) < N_LAYERS
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("레이어:", N_LAYERS, "Query 헤드/레이어:", N_HEADS)


In [ ]:
def make_token_group(texts, n_tokens):
    ids = []
    for text in texts:
        ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
        if tokenizer.eos_token_id is not None:
            ids.append(tokenizer.eos_token_id)
        if len(ids) >= n_tokens:
            return ids[:n_tokens]
    raise ValueError("평가 묶음의 토큰 수가 부족합니다.")

screen_tokens = {
    lang: make_token_group(samples["eval_groups"][lang][0], SCREEN_TOKENS)
    for lang in ("ko", "en")
}
validation_tokens = {
    (lang, group): make_token_group(
        samples["eval_groups"][lang][group - 1], VALIDATION_TOKENS
    )
    for lang in ("ko", "en") for group in (2, 3)
}

def mean_nll(token_ids, layer=None, head=None):
    hook = None
    if layer is not None:
        projection = model.model.layers[layer].self_attn.o_proj

        def mask(_module, args):
            x = args[0].clone()
            if x.shape[-1] != N_HEADS * HEAD_DIM:
                raise ValueError(f"예상과 다른 attention 출력 폭: {x.shape[-1]}")
            if head is None:
                x.zero_()
            else:
                start = head * HEAD_DIM
                x[..., start:start + HEAD_DIM] = 0
            return (x,)

        hook = projection.register_forward_pre_hook(mask)
    try:
        total_nll = 0.0
        predicted_tokens = 0
        for start in range(0, len(token_ids), SEQ_LEN):
            chunk = token_ids[start:start + SEQ_LEN]
            if len(chunk) != SEQ_LEN:
                raise ValueError("평가 토큰 길이가 512의 배수가 아닙니다.")
            input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
            with torch.inference_mode():
                output = model(input_ids=input_ids, labels=input_ids)
            count = SEQ_LEN - 1
            total_nll += output.loss.item() * count
            predicted_tokens += count
            del input_ids, output
        return total_nll / predicted_tokens
    finally:
        if hook is not None:
            hook.remove()


## 1. 모든 헤드에 대한 기울기 측정

게이트 값은 모두 1이므로 이 단계에서 헤드를 끄지 않습니다. 모델 가중치는 고정하고 게이트에 대해서만 역전파합니다. 한 언어의 4개 구간 기울기를 평균해 헤드별 국소 중요도를 얻습니다. 기울기가 `NaN` 또는 `Inf`이면 그 결과를 사용하지 않고 오류를 냅니다.


In [ ]:
def gate_gradients(token_ids):
    gates = torch.ones(N_LAYERS, N_HEADS, device="cuda:0", requires_grad=True)
    hooks = []
    for layer in range(N_LAYERS):
        projection = model.model.layers[layer].self_attn.o_proj

        def gate_output(_module, args, layer=layer):
            x = args[0]
            if x.shape[-1] != N_HEADS * HEAD_DIM:
                raise ValueError(f"예상과 다른 attention 출력 폭: {x.shape[-1]}")
            heads = x.reshape(*x.shape[:-1], N_HEADS, HEAD_DIM)
            scaled = heads * gates[layer].to(x.dtype).view(1, 1, N_HEADS, 1)
            return (scaled.reshape_as(x),)

        hooks.append(projection.register_forward_pre_hook(gate_output))

    try:
        total_grad = torch.zeros_like(gates)
        first_loss = None
        n_chunks = len(token_ids) // SEQ_LEN
        for start in range(0, len(token_ids), SEQ_LEN):
            chunk = token_ids[start:start + SEQ_LEN]
            input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
            output = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            if first_loss is None:
                first_loss = output.loss.item()
            grad = torch.autograd.grad(output.loss, gates)[0]
            if not torch.isfinite(grad).all():
                raise RuntimeError("게이트 기울기에 NaN/Inf가 있습니다. 이 실행 결과를 사용하지 마세요.")
            total_grad += grad.detach()
            del input_ids, output, grad
        assert total_grad.abs().sum().item() > 0, "게이트 기울기가 모두 0입니다."
        return total_grad / n_chunks, first_loss
    finally:
        for hook in hooks:
            hook.remove()

grad_ko, gated_first_loss = gate_gradients(screen_tokens["ko"])
# 게이트 1이 원본 모델 출력을 바꾸지 않는지 확인합니다.
plain_first_loss = mean_nll(screen_tokens["ko"][:SEQ_LEN])
assert abs(gated_first_loss - plain_first_loss) < 1e-3

grad_en, _ = gate_gradients(screen_tokens["en"])
print("기울기 계산 및 게이트 1 기준선 확인 완료")


## 2. 전체 상위 후보와 미탐색 레이어 후보 선정

`-∂NLL/∂gate`는 게이트를 줄였을 때의 **1차 손실 증가 추정치**입니다. `S_gradient`가 큰 후보를 우선합니다. 이 값은 PPL 변화율이 아니며 최종 결론에 사용하지 않습니다. 이전에 개별 헤드를 보지 않은 레이어의 후보도 별도로 포함해 레이어 선별의 누락을 확인합니다.


In [ ]:
gradient_rows = []
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        ko = -grad_ko[layer, head].item()
        en = -grad_en[layer, head].item()
        gradient_rows.append({
            "layer": layer, "head": head,
            "ko_local_nll_increase": ko,
            "en_local_nll_increase": en,
            "S_gradient": ko - en,
            "previously_screened_layer": layer in PREVIOUSLY_SCREENED_LAYERS,
        })

ranked = sorted(
    (row for row in gradient_rows if row["ko_local_nll_increase"] > 0),
    key=lambda row: row["S_gradient"], reverse=True
)
selected = {}
for row in ranked[:TOP_GLOBAL]:
    selected[row["layer"], row["head"]] = "전체 상위"
for row in (row for row in ranked if not row["previously_screened_layer"]):
    if sum(reason == "미탐색 레이어" for reason in selected.values()) >= TOP_NEW_LAYERS:
        break
    selected.setdefault((row["layer"], row["head"]), "미탐색 레이어")
if len(selected) < TOP_GLOBAL + TOP_NEW_LAYERS:
    raise RuntimeError("한국어 기울기 중요도가 양수인 후보가 부족합니다.")

candidate_rows = [
    {**next(row for row in gradient_rows if
            (row["layer"], row["head"]) == key), "selection": reason}
    for key, reason in selected.items()
]
print("전체 헤드:", len(gradient_rows), "검증 후보:", len(candidate_rows))
display(candidate_rows)


## 3. 독립 묶음에서 헤드 하나씩 실제로 끄기

한국어·영어 각각의 **같은 묶음 내 FP16 PPL**을 기준으로 변화율을 계산합니다. 기존 결과와 비교할 때는 동일한 평가 묶음과 모델 조건만 비교하세요. 두 검증 묶음의 방향과 차이를 함께 봅니다.


In [ ]:
baseline = {key: mean_nll(ids) for key, ids in validation_tokens.items()}
validation_rows = []
for candidate in candidate_rows:
    layer, head = candidate["layer"], candidate["head"]
    for group in (2, 3):
        changes = {
            lang: 100 * math.expm1(
                mean_nll(validation_tokens[lang, group], layer, head)
                - baseline[lang, group]
            )
            for lang in ("ko", "en")
        }
        validation_rows.append({
            "layer": layer, "head": head, "selection": candidate["selection"],
            "group": group, "ko_change_pct": changes["ko"],
            "en_change_pct": changes["en"],
            "S_pp": changes["ko"] - changes["en"],
        })

summary_rows = []
for candidate in candidate_rows:
    pair = [row for row in validation_rows if
            (row["layer"], row["head"]) ==
            (candidate["layer"], candidate["head"])]
    summary_rows.append({
        "layer": candidate["layer"], "head": candidate["head"],
        "selection": candidate["selection"],
        "mean_ko_change_pct": sum(row["ko_change_pct"] for row in pair) / 2,
        "mean_en_change_pct": sum(row["en_change_pct"] for row in pair) / 2,
        "mean_S_pp": sum(row["S_pp"] for row in pair) / 2,
        "positive_ko_groups": sum(row["ko_change_pct"] > 0 for row in pair),
        "positive_S_groups": sum(row["S_pp"] > 0 for row in pair),
    })
summary_rows.sort(key=lambda row: row["mean_S_pp"], reverse=True)
display(summary_rows)


## 4. CSV 브라우저 다운로드

첫 셀은 CSV 4개를 Colab 런타임에 생성합니다. 이어지는 다운로드 셀을 **하나씩 실행**하고 각 파일의 브라우저 저장이 끝났는지 확인하세요. 이미 결과 계산을 마친 런타임에서는 이 부분부터 실행하면 됩니다. GPU 평가를 다시 실행할 필요가 없습니다.


In [ ]:
def save_csv(name, rows):
    with open(name, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

run_info = [{
    "model_id": MODEL_ID,
    "sample_sha256": hashlib.sha256(sample_bytes).hexdigest(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "screen_group": 1, "screen_tokens_per_language": SCREEN_TOKENS,
    "validation_groups": "2,3", "validation_tokens_per_group": VALIDATION_TOKENS,
    "previously_screened_layers": ",".join(map(str, sorted(PREVIOUSLY_SCREENED_LAYERS))),
    "top_global": TOP_GLOBAL, "top_new_layers": TOP_NEW_LAYERS,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}]
for name, rows in (
    ("gradient_all_heads.csv", gradient_rows),
    ("gradient_validation_by_group.csv", validation_rows),
    ("gradient_validation_summary.csv", summary_rows),
    ("gradient_run_info.csv", run_info),
):
    save_csv(name, rows)
print("CSV 4개 생성 완료. 아래 다운로드 셀을 하나씩 실행하세요.")


In [ ]:
files.download("gradient_all_heads.csv")


In [ ]:
files.download("gradient_validation_by_group.csv")


In [ ]:
files.download("gradient_validation_summary.csv")


In [ ]:
files.download("gradient_run_info.csv")
